# Lesson 4 — Multi-Qubit Systems and Tensor Products

**Quantum Computing with Qiskit**

This notebook accompanies Lesson 4. Work through the cells in order. Every cell is meant to be run; modify the code freely and re-run to experiment.

**Topics covered:**
- Tensor product of vectors and gates
- Qiskit's little-endian qubit ordering and the Kronecker product
- `Operator` class for extracting unitary matrices
- The swap test: estimating state overlap with a Fredkin gate
- Circuit width and depth: definitions, measurement, and trade-offs
- GHZ state: linear vs tree-based preparation

In [ ]:
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc --quiet

In [ ]:
import qiskit
import qiskit_aer
import qiskit_ibm_runtime

print("Qiskit:             ", qiskit.__version__)
print("Qiskit Aer:         ", qiskit_aer.__version__)
print("Qiskit IBM Runtime: ", qiskit_ibm_runtime.__version__)

## 1. Tensor Product of Vectors

For an $n$-qubit system, the state space is $\mathbb{C}^{2^n}$. The tensor product combines two column vectors entry-wise:

$$\begin{pmatrix}a\\b\end{pmatrix} \otimes \begin{pmatrix}c\\d\end{pmatrix} = \begin{pmatrix}ac\\ad\\bc\\bd\end{pmatrix}$$

Qiskit uses little-endian ordering: qubit 0 is the rightmost (least significant) bit. For two qubits, use `np.kron(q1_state, q0_state)`.

In [ ]:
import numpy as np

# Single-qubit basis states
ket0 = np.array([1, 0])
ket1 = np.array([0, 1])

# Two-qubit basis states (Qiskit convention: q1 ⊗ q0)
ket00 = np.kron(ket0, ket0)   # |q1=0, q0=0⟩
ket01 = np.kron(ket0, ket1)   # |q1=0, q0=1⟩
ket10 = np.kron(ket1, ket0)   # |q1=1, q0=0⟩
ket11 = np.kron(ket1, ket1)   # |q1=1, q0=1⟩

print("|00⟩:", ket00)   # [1, 0, 0, 0]
print("|01⟩:", ket01)   # [0, 1, 0, 0]
print("|10⟩:", ket10)   # [0, 0, 1, 0]
print("|11⟩:", ket11)   # [0, 0, 0, 1]

### Building product states in Qiskit

A **product state** is one that can be written as $|\psi\rangle \otimes |\phi\rangle$. The two subsystems are independent.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Build |+⟩ ⊗ |0⟩: qubit 1 in |+⟩, qubit 0 in |0⟩
plus = np.array([1, 1]) / np.sqrt(2)
ket0 = np.array([1, 0])

sv_manual = np.kron(plus, ket0)   # q1 ⊗ q0
print("Manual |+0⟩:  ", sv_manual.round(3))

# Same state using a circuit: H on qubit 1
qc = QuantumCircuit(2)
qc.h(1)

sv_qiskit = Statevector.from_label('00').evolve(qc)
print("Qiskit |+0⟩:  ", sv_qiskit.data.round(3))
print("Match:        ", np.allclose(sv_manual, sv_qiskit.data))

Non-zero amplitudes are at indices 0 ($|q_1{=}0, q_0{=}0\rangle$) and 2 ($|q_1{=}1, q_0{=}0\rangle$): qubit 0 is always 0, qubit 1 is in equal superposition — the product structure is visible directly in the statevector.

## 2. Gates on Multi-Qubit Systems

Applying a single-qubit gate $G$ to qubit 0 of a two-qubit system gives the full matrix $I \otimes G$. Applying it to qubit 1 gives $G \otimes I$.

The `Operator` class extracts the unitary matrix of any circuit, letting you verify this directly.

In [ ]:
from qiskit.quantum_info import Operator

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
I = np.eye(2)

# H on qubit 0: full matrix should be I ⊗ H
qc_q0 = QuantumCircuit(2)
qc_q0.h(0)

matrix_q0 = Operator(qc_q0).data
print("H on qubit 0 (Qiskit):")
print(matrix_q0.round(3))
print()
print("I ⊗ H (expected):")
print(np.kron(I, H).round(3))
print()
print("Match:", np.allclose(matrix_q0, np.kron(I, H)))

In [ ]:
# H on qubit 1: full matrix should be H ⊗ I
qc_q1 = QuantumCircuit(2)
qc_q1.h(1)

matrix_q1 = Operator(qc_q1).data
print("H on qubit 1 (Qiskit):")
print(matrix_q1.round(3))
print()
print("H ⊗ I (expected):")
print(np.kron(H, I).round(3))
print()
print("Match:", np.allclose(matrix_q1, np.kron(H, I)))

In [ ]:
# H on both qubits: full matrix should be H ⊗ H
qc_both = QuantumCircuit(2)
qc_both.h(0)
qc_both.h(1)

op = Operator(qc_both)
expected = np.kron(H, H)
print("H ⊗ H match:", np.allclose(op.data, expected))

## 3. The Swap Test

The swap test estimates the squared overlap $|\langle\psi|\phi\rangle|^2$ between two states using one ancilla qubit and a controlled-SWAP (Fredkin) gate.

**Circuit:**
1. H on the ancilla (q0)
2. Controlled-SWAP: ancilla controls whether q1 and q2 are swapped
3. H on the ancilla
4. Measure the ancilla

**Result:**
$$P(\text{ancilla} = 0) = \frac{1 + |\langle\psi|\phi\rangle|^2}{2}$$
$$|\langle\psi|\phi\rangle|^2 = 2P(0) - 1$$

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def swap_test(angle1, angle2, shots=8192):
    """
    Estimate |⟨ψ|φ⟩|² between two single-qubit states prepared by Ry rotations.
      q0: ancilla
      q1: |ψ⟩ = Ry(angle1)|0⟩
      q2: |φ⟩ = Ry(angle2)|0⟩
    Returns (estimated squared overlap, circuit).
    """
    qc = QuantumCircuit(3, 1)

    # Prepare the two states
    qc.ry(angle1, 1)
    qc.ry(angle2, 2)

    # Swap test
    qc.h(0)
    qc.cswap(0, 1, 2)   # Fredkin gate: controlled-SWAP
    qc.h(0)

    # Measure only the ancilla
    qc.measure(0, 0)

    sim = AerSimulator()
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    p0 = counts.get('0', 0) / shots
    return 2 * p0 - 1, qc

# Test 1: identical states — overlap should be 1
overlap, _ = swap_test(np.pi / 3, np.pi / 3)
print(f"Identical states:  |⟨ψ|φ⟩|² = {overlap:.3f}  (expected: 1.000)")

# Test 2: orthogonal states |0⟩ and |1⟩ — overlap should be 0
overlap, _ = swap_test(0, np.pi)
print(f"Orthogonal states: |⟨ψ|φ⟩|² = {overlap:.3f}  (expected: 0.000)")

# Test 3: known angle difference
theta = np.pi / 4
overlap, _ = swap_test(0, theta)
expected = np.cos(theta / 2) ** 2   # |⟨0|Ry(θ)|0⟩|² = cos²(θ/2)
print(f"Angle diff π/4:    |⟨ψ|φ⟩|² = {overlap:.3f}  (expected: {expected:.3f})")

In [ ]:
# Draw the swap test circuit for one example
_, qc_example = swap_test(np.pi / 3, np.pi / 4)
qc_example.draw('mpl')

### Scanning the overlap across angles

In [ ]:
import matplotlib.pyplot as plt

angles = np.linspace(0, np.pi, 20)
measured = []
theoretical = []

for angle in angles:
    ov, _ = swap_test(0, angle, shots=4096)
    measured.append(ov)
    theoretical.append(np.cos(angle / 2) ** 2)

plt.figure(figsize=(7, 4))
plt.plot(angles / np.pi, theoretical, label='Theoretical $\\cos^2(\\theta/2)$')
plt.scatter(angles / np.pi, measured, label='Swap test estimate', zorder=5)
plt.xlabel('$\\theta / \\pi$')
plt.ylabel('$|\\langle 0 | R_y(\\theta) | 0 \\rangle|^2$')
plt.title('Swap Test: Estimated vs Theoretical Overlap')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Circuit Width and Depth

- **Width** = number of qubits (`qc.num_qubits`)
- **Depth** = length of the longest sequential gate chain (`qc.depth()`); gates on disjoint qubits run in parallel and count as one layer
- **Gate count** = total number of gates (`qc.size()`)

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(4)
qc.h(0)
qc.h(1)
qc.h(2)
qc.h(3)
qc.cx(0, 1)
qc.cx(2, 3)   # independent of cx(0,1): runs in the same layer
qc.cx(1, 2)   # depends on both previous cx gates

print("Width (qubits):", qc.num_qubits)
print("Depth:         ", qc.depth())
print("Gate count:    ", qc.size())

qc.draw('mpl')

7 gates but depth 3: the four H gates form one layer, `cx(0,1)` and `cx(2,3)` form a second layer (disjoint qubits), and `cx(1,2)` forms the third.

### GHZ state: linear vs tree-based preparation

The $n$-qubit GHZ state $\frac{1}{\sqrt{2}}(|0\rangle^{\otimes n} + |1\rangle^{\otimes n})$ demonstrates a clear depth trade-off.

In [ ]:
n = 8

# Linear: H then a chain of n-1 CNOT gates — depth n
qc_linear = QuantumCircuit(n)
qc_linear.h(0)
for i in range(n - 1):
    qc_linear.cx(i, i + 1)

print("Linear GHZ:")
print("  Depth:     ", qc_linear.depth())
print("  Gate count:", qc_linear.size())

In [ ]:
# Tree: each round doubles the entangled qubits — depth log₂(n) + 1
qc_tree = QuantumCircuit(n)
qc_tree.h(0)

step = 1
while step < n:
    for i in range(0, n - step, 2 * step):
        qc_tree.cx(i, i + step)
    step *= 2

print("Tree GHZ:")
print("  Depth:     ", qc_tree.depth())
print("  Gate count:", qc_tree.size())

In [ ]:
# Draw both circuits
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
qc_linear.draw('mpl', ax=axes[0])
axes[0].set_title(f'Linear GHZ (depth {qc_linear.depth()})')
qc_tree.draw('mpl', ax=axes[1])
axes[1].set_title(f'Tree GHZ (depth {qc_tree.depth()})')
plt.tight_layout()
plt.show()

### Depth after transpilation to real hardware

Abstract depth ignores hardware connectivity. SWAP insertions (each costing 3 CNOTs) for non-adjacent qubits can reverse the ordering.

In [ ]:
from qiskit import transpile
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

fake = FakeBrisbane()

t_linear = transpile(qc_linear, fake, optimization_level=3)
t_tree   = transpile(qc_tree,   fake, optimization_level=3)

print("After transpilation to FakeBrisbane:")
print(f"  Linear GHZ depth: {t_linear.depth()}")
print(f"  Tree GHZ depth:   {t_tree.depth()}")
print()
print("The tree circuit requires long-range CNOTs that need SWAP insertions.")
print("The linear chain aligns with nearest-neighbor connectivity, so it transpiles more efficiently.")

## 5. Exercises

### Exercise 1: Tensor product of three qubits

Construct the state $|{+}\rangle \otimes |1\rangle \otimes |0\rangle$ (qubit 2 in $|{+}\rangle$, qubit 1 in $|1\rangle$, qubit 0 in $|0\rangle$) two ways:

1. Using `np.kron` three times.
2. Using a `QuantumCircuit(3)` with appropriate gates, then `Statevector.from_label('00').evolve(qc)` (you will need to start from the right initial state or use X and H gates).

Confirm the two vectors match.

In [ ]:
# Exercise 1: your solution here
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

ket0 = np.array([1, 0])
ket1 = np.array([0, 1])
plus = np.array([1, 1]) / np.sqrt(2)

# Method 1: numpy Kronecker products (q2 ⊗ q1 ⊗ q0)
# TODO

# Method 2: QuantumCircuit
# TODO

### Exercise 2: Gate matrix verification

Use the `Operator` class to extract the full unitary matrix for each of the following two-qubit circuits, then verify each identity using `np.allclose`:

1. `S` on qubit 0: expected matrix is $I \otimes S$.
2. `T` on qubit 1: expected matrix is $T \otimes I$.
3. `X` on qubit 0 followed by `Z` on qubit 1: expected matrix is $Z \otimes X$.

The gate matrices are: $S = \begin{pmatrix}1&0\\0&i\end{pmatrix}$, $T = \begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$.

In [ ]:
# Exercise 2: your solution here
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

I = np.eye(2)
S = np.array([[1, 0], [0, 1j]])
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])

# 1. S on qubit 0
qc1 = QuantumCircuit(2)
qc1.s(0)
# TODO: verify Operator(qc1).data == np.kron(I, S)

# 2. T on qubit 1
# TODO

# 3. X on qubit 0, Z on qubit 1
# TODO

### Exercise 3: Swap test for multi-qubit states

The swap test works for multi-qubit states too. Extend `swap_test` to compare two 2-qubit states.

1. Build a function `swap_test_2q(angles1, angles2, shots=8192)` where `angles1 = (theta0, theta1)` prepares qubit 1 with `Ry(theta0)` and qubit 2 with `Ry(theta1)`, and `angles2` prepares the second pair similarly. Use qubits 0 as ancilla, 1 and 2 as the first state, and 3 and 4 as the second state. Apply `cswap` on pairs (0,1,3) and (0,2,4).
2. Test with two identical states and two orthogonal states.
3. Verify analytically: for two-qubit product states $|\psi_1\rangle|\psi_2\rangle$ and $|\phi_1\rangle|\phi_2\rangle$, the overlap is $|\langle\psi_1|\phi_1\rangle|^2 \cdot |\langle\psi_2|\phi_2\rangle|^2$.

In [ ]:
# Exercise 3: your solution here
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def swap_test_2q(angles1, angles2, shots=8192):
    """
    Estimate |⟨ψ|φ⟩|² between two 2-qubit product states.
    angles1 = (a0, a1): Ry rotations for the first state (qubits 1, 2)
    angles2 = (b0, b1): Ry rotations for the second state (qubits 3, 4)
    Ancilla: qubit 0
    """
    qc = QuantumCircuit(5, 1)
    # TODO: prepare states, apply swap test, measure ancilla
    pass

# TODO: test with identical and orthogonal states

### Exercise 4: Scaling depth with system size

Compare the linear and tree GHZ preparation depths for $n = 2, 4, 8, 16, 32$ qubits (abstract, without transpilation).

1. Compute and print the depth for each $n$ and each method.
2. Plot depth vs $n$ for both methods on the same axes.
3. Fit the linear method to $an + b$ and the tree method to $a\log_2(n) + b$ and overlay the fits.

In [ ]:
# Exercise 4: your solution here
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit

n_values = [2, 4, 8, 16, 32]
depths_linear = []
depths_tree = []

for n in n_values:
    # Linear GHZ
    qc_lin = QuantumCircuit(n)
    qc_lin.h(0)
    for i in range(n - 1):
        qc_lin.cx(i, i + 1)
    depths_linear.append(qc_lin.depth())

    # Tree GHZ
    qc_tr = QuantumCircuit(n)
    qc_tr.h(0)
    step = 1
    while step < n:
        for i in range(0, n - step, 2 * step):
            qc_tr.cx(i, i + step)
        step *= 2
    depths_tree.append(qc_tr.depth())

print(f"{'n':>5}  {'Linear':>8}  {'Tree':>8}")
for n, dl, dt in zip(n_values, depths_linear, depths_tree):
    print(f"{n:>5}  {dl:>8}  {dt:>8}")

# TODO: plot depth vs n for both methods

## Summary

In this lesson you:

- Computed tensor products of column vectors using `np.kron`, following Qiskit's little-endian convention (`q1 ⊗ q0`).
- Verified that applying a single-qubit gate $G$ to qubit 0 gives the full matrix $I \otimes G$, and to qubit 1 gives $G \otimes I$, using the `Operator` class.
- Implemented the swap test with `qc.cswap(control, t1, t2)` and confirmed $P(0) = (1 + |\langle\psi|\phi\rangle|^2)/2$.
- Measured circuit width and depth with `qc.num_qubits` and `qc.depth()`, and observed that independent gates on disjoint qubits count as one layer.
- Compared linear-depth and log-depth GHZ preparation, and found that after transpilation to a real device the log-depth circuit can become deeper due to SWAP insertions.

**Lesson 5** introduces entanglement: the Bell states, their measurement correlations, GHZ and W states, and the CHSH inequality.